# Prepared-data experiment — execution track 0

This notebook runs the archived, checksum-verified experimental implementation. It uses persistent output directories and resumes completed stages by their stored signatures.

**Prerequisites:** a clone of this repository in `/content/beyond-generating-cell`, the original prepared KITTI360Pose `cells/` and `poses/` folders in Google Drive, and sufficient compute resources.


In [ ]:
from pathlib import Path
import sys, subprocess
from google.colab import drive
drive.mount('/content/drive')

# Clone the repository into /content and configure the prepared-data location below.
ROOT=Path('/content/beyond-generating-cell')
STAGE=Path('/content/bgc_historical_verified')
DRIVE=Path('/content/drive/MyDrive')
DATA=DRIVE/'KITTI360Pose_prepared'  # Directory containing cells/ and poses/
OUT=DRIVE/'beyond_generating_cell_results'
EXISTING=DRIVE/'beyond_generating_cell_previous_results'
assert (ROOT/'scripts/run_historical.py').is_file(), 'Extract the full repository to ROOT first'
assert (DATA/'cells').is_dir() and (DATA/'poses').is_dir(), 'Missing original prepared dataset'
OUT.mkdir(parents=True,exist_ok=True)
print('ROOT',ROOT,'DATA',DATA,'OUT',OUT)

## Check original source identity and run historical unit tests

The locked source ZIP is extracted once to local Colab storage; all existing Drive scientific results remain untouched.

In [ ]:
def launch(mode, *args):
    command=[sys.executable, str(ROOT/'scripts/run_historical.py'), mode, '--stage', str(STAGE), *map(str,args)]
    print('Executing:', ' '.join(command),flush=True)
    subprocess.run(command,cwd=ROOT,check=True)
launch('core-tests')
launch('executor-tests')

## Create/verify a durable experimental plan

This is the only Track 0 action needed before Track 1 runs. Source-tree and dataset hashes are checked by the historical runner. It can take time to fingerprint the dataset. If previous completed results are absent, leave `--existing` out and expect all jobs to be scheduled.

In [ ]:
args=['--dataset',str(DATA),'--out',str(OUT)]
if EXISTING.is_dir():
    args += ['--existing',str(EXISTING)]
launch('plan',*args)
assert (OUT/'PLAN.json').is_file(), 'No durable PLAN.json generated'

## Execute / resume track 0

If a job fails, read its per-job log, fix only the environment or missing input, and rerun **this** cell in the same Drive output directory. All original signature checks remain active; reruns do not require a Colab server restart.

In [ ]:
launch('track','--out',str(OUT),'--track','0')

## Status (read-only)

In [ ]:
launch('status','--out',str(OUT))

## Finalize only after both tracks are finished

For Track 0 or Track 1: finalization uses the original historical runner; incomplete checks fail closed. Do not publish a merged archive with failed jobs.

In [ ]:
# Only run when both tracks completed successfully. This may take time.
# launch('finalize','--out',str(OUT))
print('When both tracks pass: uncomment the finalize command above.')